In [0]:
import base64

import psycopg2
from psycopg2.extras import RealDictCursor

In [0]:
# add widget for pasting in the Lakebase URL
dbutils.widgets.text("lakebase_url", "", "Lakebase URL (optional)")

In [0]:
def get_lakebase_url():
    """Return the Lakebase connection URL from the widget or the secret scope.

    Secret values are base64-encoded by the Databricks SDK, so we decode them — identical to how the app's lakebase.py resolves the URL.
    """
    pasted = dbutils.widgets.get("lakebase_url").strip()
    if pasted:
        return pasted
    
    from databricks.sdk import WorkspaceClient

    secret = WorkspaceClient().secrets.get_secret(scope="database", key="lakebase-url")
    return base64.b64decode(secret.value).decode("utf-8")

In [0]:
URL = get_lakebase_url()

SCHEMA = [
    """
    CREATE TABLE IF NOT EXISTS tickets (
        ticket_id   BIGSERIAL PRIMARY KEY,
        title       TEXT NOT NULL,
        status      TEXT NOT NULL DEFAULT 'open'
                    CHECK (status IN ('open','in_progress','resolved')),
        priority    TEXT NOT NULL DEFAULT 'medium'
                    CHECK (priority IN ('low','medium','high','urgent')),
        category    TEXT NOT NULL DEFAULT 'general',
        created_by  TEXT NOT NULL,
        created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS ticket_messages (
        message_id    BIGSERIAL PRIMARY KEY,
        ticket_id     BIGINT NOT NULL REFERENCES tickets(ticket_id) ON DELETE CASCADE,
        message_text  TEXT NOT NULL,
        author        TEXT NOT NULL,
        created_at    TIMESTAMPTZ NOT NULL DEFAULT now()
    )
    """,
    "CREATE INDEX IF NOT EXISTS idx_messages_ticket ON ticket_messages(ticket_id)",
    "CREATE INDEX IF NOT EXISTS idx_tickets_status  ON tickets(status)"
]

SAMPLE_DATA = [
    {
        "title": "Cannot log in to the analytics dashboard",
        "status": "open", "priority": "high", "category": "bug",
        "created_by": "maria.lopez@example.com",
        "messages": [
            ("maria.lopez@example.com", "I get a 500 error right after entering my password."),
            ("support.agent@example.com", "Thanks Maria — which browser are you using?"),
            ("maria.lopez@example.com", "Chrome on macOS, latest version."),
        ],
    },
    {
        "title": "Add dark mode to the reporting UI",
        "status": "in_progress", "priority": "medium", "category": "feature",
        "created_by": "james.chen@example.com",
        "messages": [
            ("james.chen@example.com", "A dark theme would really help for late-night reviews."),
            ("product.team@example.com", "Good idea — added to the current sprint."),
        ],
    },
    {
        "title": "Invoice shows the wrong tax amount",
        "status": "open", "priority": "urgent", "category": "billing",
        "created_by": "amir.khan@example.com",
        "messages": [
            ("amir.khan@example.com", "March invoice charged 20% tax instead of 10%."),
            ("billing@example.com", "Investigating — we'll issue a corrected invoice."),
        ],
    },
    {
        "title": "How do I export my data to CSV?",
        "status": "resolved", "priority": "low", "category": "question",
        "created_by": "sofia.rossi@example.com",
        "messages": [
            ("sofia.rossi@example.com", "Is there a way to export a table to CSV?"),
            ("support.agent@example.com", "Yes — use the Export button in the top-right of any table."),
            ("sofia.rossi@example.com", "Found it, thank you!"),
        ],
    },
    {
        "title": "Reset password link expired too quickly",
        "status": "in_progress", "priority": "medium", "category": "account",
        "created_by": "liam.murphy@example.com",
        "messages": [
            ("liam.murphy@example.com", "The reset link expired before I could use it."),
            ("support.agent@example.com", "We've extended the link lifetime to 24 hours; please retry."),
        ],
    },
]

In [0]:
# Create schema, clear old rows, insert the sample set — one transaction
conn = psycopg2.connect(URL, cursor_factory=RealDictCursor)
try:
    with conn.cursor() as cur:

        # create tables and indexes
        for stmt in SCHEMA:
            cur.execute(stmt)
    
        cur.execute("DELETE FROM tickets") # messages cascade via the FK

        # insert data
        for t in SAMPLE_DATA:
            cur.execute(
                """
                INSERT INTO tickets (title, status, priority, category, created_by)
                VALUES (%s, %s, %s, %s, %s)
                RETURNING ticket_id
                """,
                (t["title"], t["status"], t["priority"], t["category"], t["created_by"])
            )
            ticket_id = cur.fetchone()["ticket_id"]
            for author, text in t["messages"]:
                cur.execute(
                    """
                    INSERT INTO ticket_messages (ticket_id, message_text, author)
                    VALUES (%s, %s, %s)
                    """,
                    (ticket_id, text, author)
                )
            print(f"Seeded ticket {ticket_id}: {t['title']} ({len(t['messages'])} messages)")

    conn.commit()
    print("\nCommitted.")
        
finally:
    conn.close()

In [0]:
# Verify what landed in Lakebase
conn = psycopg2.connect(URL, cursor_factory=RealDictCursor)
try:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) AS n FROM tickets")
        n_tickets = cur.fetchone()["n"]
        cur.execute("SELECT COUNT(*) AS n FROM ticket_messages")
        n_msgs = cur.fetchone()["n"]

        cur.execute(
            """
            SELECT
                t.ticket_id, t.status, t.priority, t.category, t.title, COUNT(m.message_id) AS message_count
            FROM tickets t
            LEFT JOIN ticket_messages m
                ON t.ticket_id = m.ticket_id
            GROUP BY t.ticket_id
            ORDER BY t.ticket_id
            """
        )
        rows = cur.fetchall()
finally:
    conn.close()

In [0]:
print(f"Total tickets: {n_tickets}   Total messages: {n_msgs}\n")
for r in rows:
    print(f"  #{r['ticket_id']:>3}  [{r['status']:<12}] {r['priority']:<6} "
          f"{r['category']:<9} {r['message_count']} msgs  ·  {r['title']}")

In [0]:
display(spark.createDataFrame(
    [(r["ticket_id"], r["status"], r["priority"], r["category"],
      r["title"], r["message_count"]) for r in rows],
    ["ticket_id", "status", "priority", "category", "title", "message_count"],
))